In [ ]:
from torchgeo.trainers import PixelwiseRegressionTask
import torch
import pytorch_lightning as pl
import numpy as np
import rasterio
import cv2
import logging
from typing import List
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import torch.nn as nn
import os
from utils.model import LSTNowcaster
from utils.data.TiledLandsatDataModule import TiledLandsatDataModule

os.environ["WANDB_NOTEBOOK_NAME"] = "TrainUNet-Basic.ipynb"
os.environ["WANDB_DIR"] = "./wandb"
os.environ["WANDB_CACHE_DIR"] = "/work/ubh496/.cache/wandb"
os.environ["WANDB_CONFIG_DIR"] = "/work/ubh496/.config/wandb"
os.environ["WANDB_DATA_DIR"] = "/work/ubh496/.cache/wandb-data"
os.environ["WANDB_ARTIFACT_DIR"] = "./artifacts"

config = {
    "experiment_name": "Batch Testing",
    "debug": True,
    "by_city": False,
    "months_ahead": 0,
    "tile_size": 512,
    "learning_rate": 1e-4,
    "model": "unet",
    "backbone": "resnet50",
    "dataset": "pure_landsat",
    "epochs": 25,
    "batch_size": 8,
    "pretrained_weights": True,
    "deterministic": True,
    "in_channels": 5
}

wandb_logger = WandbLogger(
    project="heat-island",  # your project name
    name=config['experiment_name'],  # name of this particular run
    log_model="all",  # log model checkpoints
    save_code=True,
    save_dir="./wandb",  # where to save the logs locally
)
wandb_logger.log_hyperparams(config)
if config["dataset"] == "pure_landsat":
    data_module = TiledLandsatDataModule(
        data_dir="./Data",
        monthsAhead=config["months_ahead"],
        batch_size=config["batch_size"],
        num_workers=5,
        byCity=config["by_city"],
        debug=config["debug"],
        tile_size=config["tile_size"],  # Adjust based on your GPU memory
        tile_overlap=0.2  # 20% overlap between tiles
    )
    data_module.setup()

checkpoint_callback = ModelCheckpoint(dirpath="./wandb/heat-island/checkpoints",monitor="val_rmse_F", mode="min")

# Initialize trainer with explicit steps
trainer = pl.Trainer(
    max_epochs=config["epochs"],
    gradient_clip_val=0.5,
    log_every_n_steps=10,
    enable_progress_bar=True,
    enable_model_summary=False,
    deterministic=config["deterministic"],
    num_sanity_val_steps=2,
    reload_dataloaders_every_n_epochs=1,
    logger=wandb_logger,
    callbacks=[checkpoint_callback]
)

wandb: Currently logged in as: jesus-guerrero (jesus-guerrero-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Preparing scene by scene...: 100%|██████████| 650/650 [00:01<00:00, 513.94it/s]
/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /work/ubh496/.conda/envs/ml3/lib/python3.10/site-pac ...
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [2]:
from utils.model import LSTNowcaster
model = LSTNowcaster(model=config["model"], backbone=config["backbone"], in_channels=config["in_channels"], learning_rate=config["learning_rate"], pretrained_weights=config["pretrained_weights"])

/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/torch/hub.py:846: UserWarning: TORCH_MODEL_ZOO is deprecated, please use env TORCH_HOME instead
  warnings.warn(


In [3]:
trainer.fit(model=model, datamodule=data_module)

Preparing scene by scene...: 100%|██████████| 650/650 [00:00<00:00, 3879.99it/s]
/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /work/ubh496/heat-island-test/wandb/heat-island/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=25` reached.


In [4]:
# After training, load the best checkpoint for testing
best_model_path = checkpoint_callback.best_model_path
print(f"Best model path: {best_model_path}")

if best_model_path:
    # Load best checkpoint
    best_model = LSTNowcaster.load_from_checkpoint(
        checkpoint_path=best_model_path,
        model=config["model"],
        backbone=config["backbone"],
        in_channels=config["in_channels"],
        learning_rate=config["learning_rate"],
        pretrained_weights=config["pretrained_weights"]
    )
    
    # Test using the best model
    trainer.test(model=best_model, datamodule=data_module)
else:
    print("No checkpoint found, testing with the current model state")
    trainer.test(model=model, datamodule=data_module)

Best model path: /work/ubh496/heat-island-test/wandb/heat-island/checkpoints/epoch=23-step=7368.ckpt


Preparing scene by scene...: 100%|██████████| 650/650 [00:00<00:00, 1572.43it/s]
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_rmse_F        │    13.140706062316895     │
└───────────────────────────┴───────────────────────────┘